In [3]:
# ============================================================
# OPTIMIZED Centris Photos Downloader (Colab) ✅ FAST + ADDRESS FIX
# - keep-only (no junk)
# - dedupe (pick biggest per visual cluster)
# - address extraction robust (handles: "#, Street, City (Neighborhood)")
# - if address not found => address_not_found
# - ROOM LABELS:
#     * default fast mode => room_unknown (no ML)
#     * optional enable CLIP => slower but adds kitchen/bedroom/etc.
# - ZIP name: listingNo_address_downloadDateTime.zip
# ============================================================

# ---- SETTINGS YOU CAN CHANGE ----
ENABLE_CLIP_ROOM_LABELS = False   # ✅ fastest: False   |  room labels: True
MAX_GALLERY_STEPS = 55            # try 45–70 depending on listing size
EXTRA_SCROLLS = 4                 # fewer scrolls = faster
HAMMING_MAX = 7
MIN_GOOD_W, MIN_GOOD_H = 700, 450
MIN_GOOD_BYTES = 60_000

# --- 1) Install deps ---
!apt-get -qq update
!apt-get -qq install -y \
  libatk-bridge2.0-0 libatk1.0-0 libcups2 libxkbcommon0 \
  libxcomposite1 libxdamage1 libxrandr2 libgbm1 libasound2 \
  libpangocairo-1.0-0 libpango-1.0-0 libgtk-3-0 \
  libnss3 libxshmfence1 fonts-liberation

!pip -q install --upgrade playwright nest_asyncio pillow
!playwright install chromium

import re, json, hashlib, zipfile, shutil, unicodedata
from pathlib import Path
from datetime import datetime

import nest_asyncio
nest_asyncio.apply()

from PIL import Image
from playwright.async_api import async_playwright
from google.colab import files

# Optional CLIP install only if enabled (saves time)
if ENABLE_CLIP_ROOM_LABELS:
    !pip -q install --upgrade transformers accelerate torch
    import torch
    from transformers import pipeline


# ======================
# USER INPUT
# ======================
user_in = input("Enter Centris listing NUMBER (e.g., 12345678) OR full Centris URL: ").strip()

def extract_listing_number(s: str) -> str | None:
    if re.fullmatch(r"\d{6,}", s):
        return s
    m = re.search(r"/(\d{6,})(?:\?|$)", s)
    return m.group(1) if m else None

listing_no = extract_listing_number(user_in) or "UNKNOWN"

candidate_urls = []
if user_in.startswith("http://") or user_in.startswith("https://"):
    candidate_urls = [user_in]
else:
    candidate_urls = [
        f"https://www.centris.ca/fr/propriete~a-vendre~{listing_no}?nocontext=true",
        f"https://www.centris.ca/fr/propriete~a-vendre~{listing_no}",
    ]


# ======================
# OUTPUT STRUCTURE (UNIQUE PER RUN)
# ======================
download_dt = datetime.now().strftime("%Y%m%d_%H%M%S")
BASE_DIR = Path(f"centris_{listing_no}_{download_dt}")
RAW_DIR = BASE_DIR / "_raw"
BASE_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)


# ======================
# CLIP SETUP (only if enabled)
# ======================
ROOM_LABELS = [
    "kitchen","bathroom","bedroom","living_room","dining_room","basement",
    "laundry_room","garage","office","hallway","stairs","closet",
    "exterior_front","exterior_back","backyard","aerial","neighborhood","floorplan","other"
]
LABEL_ALIAS = {"laundry_room":"laundry","living_room":"living","dining_room":"dining","exterior_front":"front","exterior_back":"back"}
MIN_ROOM_SCORE = 0.22

if ENABLE_CLIP_ROOM_LABELS:
    device = 0 if torch.cuda.is_available() else -1
    room_classifier = pipeline(
        "zero-shot-image-classification",
        model="openai/clip-vit-base-patch32",
        device=device
    )


# ======================
# HELPERS
# ======================
def sanitize_filename(s: str, max_len: int = 80) -> str:
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
    s = s.lower().strip()
    s = re.sub(r"[^\w\s-]", "", s)
    s = re.sub(r"[\s-]+", "_", s).strip("_")
    return s[:max_len] if len(s) > max_len else s

def ext_from_content_type(ct: str) -> str:
    ct = (ct or "").lower()
    if "png" in ct: return ".png"
    if "webp" in ct: return ".webp"
    return ".jpg"

def image_info(path: Path):
    b = path.stat().st_size
    try:
        with Image.open(path) as im:
            return im.size[0], im.size[1], b
    except:
        return None, None, b

def dhash(path: Path, hash_size: int = 8) -> int:
    with Image.open(path) as im:
        im = im.convert("L").resize((hash_size + 1, hash_size), Image.Resampling.LANCZOS)
        pixels = list(im.getdata())
        rows = [pixels[i*(hash_size+1):(i+1)*(hash_size+1)] for i in range(hash_size)]
        bits = []
        for row in rows:
            for col in range(hash_size):
                bits.append(1 if row[col] > row[col+1] else 0)
    h = 0
    for bit in bits:
        h = (h << 1) | bit
    return h

def hamming(a: int, b: int) -> int:
    return (a ^ b).bit_count()

def sha8_of_file(p: Path) -> str:
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()[:8]

async def accept_privacy_popup(page):
    labels = ["Accepter et continuer", "Accepter", "Tout accepter", "J’accepte", "J'accepte", "OK"]
    for lab in labels:
        try:
            btn = page.get_by_role("button", name=lab)
            if await btn.count() > 0:
                await btn.first.click(timeout=5000)
                await page.wait_for_timeout(900)
                return True
        except:
            pass
    try:
        btn = page.locator("button:has-text('Accepter')").first
        if await btn.count() > 0:
            await btn.click(timeout=5000)
            await page.wait_for_timeout(900)
            return True
    except:
        pass
    return False

async def open_gallery(page):
    for txt in ["Voir toutes les photos", "Voir les photos", "Toutes les photos", "Photos"]:
        try:
            loc = page.get_by_text(txt, exact=False)
            if await loc.count() > 0:
                await loc.first.click(timeout=5000)
                await page.wait_for_timeout(900)
                return True
        except:
            pass
    # fallback click first large image
    try:
        imgs = page.locator("img")
        n = await imgs.count()
        for i in range(min(n, 40)):
            el = imgs.nth(i)
            box = await el.bounding_box()
            if box and box["width"] > 260 and box["height"] > 180:
                await el.click(timeout=5000)
                await page.wait_for_timeout(900)
                return True
    except:
        pass
    return False

async def extract_address(page) -> str:
    """
    Robust Centris address extraction:
    - Handles "#, Street, City (Neighborhood)"
    - Returns 'address_not_found' if not found.
    """

    # 1) JSON-LD
    try:
        scripts = page.locator("script[type='application/ld+json']")
        count = await scripts.count()
        for i in range(count):
            raw = (await scripts.nth(i).inner_text()) or ""
            raw = raw.strip()
            if not raw:
                continue
            try:
                data = json.loads(raw)
            except:
                continue

            def iter_objs(x):
                if isinstance(x, dict):
                    yield x
                    if isinstance(x.get("@graph"), list):
                        for o in x["@graph"]:
                            yield from iter_objs(o)
                elif isinstance(x, list):
                    for o in x:
                        yield from iter_objs(o)

            for obj in iter_objs(data):
                addr = obj.get("address")
                if isinstance(addr, dict):
                    street = (addr.get("streetAddress") or "").strip()
                    city   = (addr.get("addressLocality") or "").strip()
                    region = (addr.get("addressRegion") or "").strip()
                    postal = (addr.get("postalCode") or "").strip()
                    if street and re.search(r"\d", street):
                        parts = [street]
                        if city: parts.append(city)
                        if region: parts.append(region)
                        if postal: parts.append(postal)
                        return ", ".join(parts)
    except:
        pass

    # 2) DOM text scans (fast)
    # Note: comma after civic number is important: "11, Place ..."
    address_like = re.compile(
        r"\b\d{1,5}\s*,\s*[^,\n]{2,80},\s*[^,\n]{2,80}(?:\s*\([^)]+\))?",
        re.IGNORECASE
    )

    for sel in ["header", "main"]:
        try:
            txt = (await page.inner_text(sel)).strip()
            m = address_like.search(txt)
            if m:
                cand = m.group(0).strip()
                if 8 <= len(cand) <= 160:
                    return cand
        except:
            pass

    # 3) Whole page scan (still ok)
    try:
        txt = (await page.inner_text("body")).strip()
        m = address_like.search(txt)
        if m:
            cand = m.group(0).strip()
            if 8 <= len(cand) <= 180:
                return cand
    except:
        pass

    return "address_not_found"

def predict_room_label_fast(_img_path: Path) -> tuple[str, float]:
    return ("room_unknown", 0.0)

def predict_room_label_clip(img_path: Path) -> tuple[str, float]:
    # Speed: downscale before classification
    tmp = img_path.with_suffix(".tmp.jpg")
    try:
        with Image.open(img_path) as im:
            im = im.convert("RGB")
            im.thumbnail((640, 640))
            im.save(tmp, quality=85)
        res = room_classifier(str(tmp), candidate_labels=ROOM_LABELS)
        best = res[0]
        label = best["label"]
        score = float(best["score"])
        if score < MIN_ROOM_SCORE:
            return ("other", score)
        label = LABEL_ALIAS.get(label, label)
        label = sanitize_filename(label, max_len=20) or "other"
        return (label, score)
    except:
        return ("other", 0.0)
    finally:
        try:
            tmp.unlink()
        except:
            pass

predict_room_label = predict_room_label_clip if ENABLE_CLIP_ROOM_LABELS else predict_room_label_fast


# ======================
# CAPTURE
# ======================
async def capture_all_images():
    seen_sha = set()
    saved = []
    selected_url = None
    captured_address = None

    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=["--no-sandbox", "--disable-dev-shm-usage"]
        )
        context = await browser.new_context(
            viewport={"width": 1400, "height": 900},
            user_agent="Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36"
        )
        page = await context.new_page()

        async def on_response(resp):
            try:
                ct = (resp.headers.get("content-type") or "").lower()
                if not ct.startswith("image/"):
                    return
                data = await resp.body()
                sha = hashlib.sha256(data).hexdigest()
                if sha in seen_sha:
                    return
                seen_sha.add(sha)

                ext = ext_from_content_type(ct)
                fname = f"raw_{len(saved)+1:04d}_{sha[:8]}{ext}"
                path = RAW_DIR / fname
                with open(path, "wb") as f:
                    f.write(data)
                saved.append(path)
            except:
                pass

        page.on("response", on_response)

        # Open page
        for url in candidate_urls:
            try:
                resp = await page.goto(url, wait_until="domcontentloaded", timeout=60000)
                await page.wait_for_timeout(1200)
                if resp and resp.ok:
                    selected_url = page.url
                    break
            except:
                pass

        if not selected_url:
            await browser.close()
            raise RuntimeError("Could not open the listing page. Provide the FULL Centris URL.")

        await accept_privacy_popup(page)
        await page.wait_for_timeout(900)  # let header render after consent

        captured_address = await extract_address(page)

        # a few scrolls to load lazy images
        for _ in range(EXTRA_SCROLLS):
            await page.mouse.wheel(0, 1700)
            await page.wait_for_timeout(380)

        await open_gallery(page)

        # step through gallery
        for _ in range(MAX_GALLERY_STEPS):
            try:
                await page.keyboard.press("ArrowRight")
            except:
                pass
            await page.wait_for_timeout(240)

        await page.wait_for_timeout(1200)
        await browser.close()

    return saved, selected_url, captured_address


raw_files, final_url, address_text = await capture_all_images()
print("Opened:", final_url)
print("Captured raw images:", len(raw_files))
print("Extracted address:", address_text)


# ======================
# CLUSTER + KEEP ONLY
# ======================
items = []
for p in raw_files:
    w, h, b = image_info(p)
    if b < MIN_GOOD_BYTES // 4:   # quick prefilter
        continue
    try:
        ph = dhash(p)
    except:
        ph = None
    items.append({"path": p, "w": w, "h": h, "bytes": b, "ph": ph})

items.sort(key=lambda x: x["bytes"], reverse=True)

clusters = []
for it in items:
    if it["ph"] is None:
        clusters.append({"rep_ph": None, "members": [it]})
        continue

    placed = False
    for c in clusters:
        if c["rep_ph"] is None:
            continue
        if hamming(it["ph"], c["rep_ph"]) <= HAMMING_MAX:
            c["members"].append(it)
            placed = True
            break
    if not placed:
        clusters.append({"rep_ph": it["ph"], "members": [it]})

keep_paths = []
for c in clusters:
    rep = max(c["members"], key=lambda x: x["bytes"])
    w, h, b = rep["w"], rep["h"], rep["bytes"]
    is_good = (b >= MIN_GOOD_BYTES) and ((w is None) or (w >= MIN_GOOD_W and h >= MIN_GOOD_H))
    if is_good:
        keep_paths.append(rep["path"])

print("Clusters:", len(clusters))
print("Keep candidates:", len(keep_paths))


# ======================
# MOVE kept, delete rest
# ======================
keep_paths = sorted(keep_paths, key=lambda p: p.stat().st_size, reverse=True)
final_paths = []
for i, p in enumerate(keep_paths, start=1):
    hash8 = sha8_of_file(p)
    label, _ = predict_room_label(p)
    ext = p.suffix.lower()
    name = f"photo_{i:03d}_{hash8}_{label}{ext}"
    dest = BASE_DIR / name
    shutil.move(str(p), str(dest))
    final_paths.append(dest)

# Cleanup raw dir quickly
for p in RAW_DIR.iterdir():
    if p.is_file():
        p.unlink()
RAW_DIR.rmdir()


# ======================
# ZIP
# ======================
address_slug = sanitize_filename(address_text) if address_text != "address_not_found" else "address_not_found"
zip_name = f"{listing_no}_{address_slug}_{download_dt}.zip"

with zipfile.ZipFile(zip_name, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in final_paths:
        z.write(str(p), arcname=p.name)

print("✅ ZIP created:", zip_name)
files.download(zip_name)


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Enter Centris listing NUMBER (e.g., 20236253) OR full Centris URL: https://www.centris.ca/fr/maison~a-vendre~montreal-pierrefonds-roxboro/20236253
Opened: https://www.centris.ca/fr/maison~a-vendre~montreal-pierrefonds-roxboro/20236253
Captured raw images: 77
Extracted address: 11, Place Jordan, Montréal (Pierrefonds-Roxboro)
Clusters: 35
Keep candidates: 27
✅ ZIP created: 20236253_11_place_jordan_montreal_pierrefonds_roxboro_20251231_030658.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>